In [50]:
# VERBA — Multilingual Translation Prototype

### Connecting Languages, Connecting People

"VERBA is an academic multilingual translation prototype..."

'VERBA is an academic multilingual translation prototype...'

In [51]:

# Install the libraries required by the translation model

%pip install -q "transformers<5" sentencepiece accelerate gradio

print("Dependencies installed successfully.")


Dependencies installed successfully.


In [42]:

from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:

# Import required libraries

import os
import sys
import time
import csv
import statistics

import torch
import transformers
import sentencepiece
import accelerate
import gradio as gr

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Select available hardware
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Display environment information
print("VERBA Environment")
print("-" * 35)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("Accelerate:", accelerate.__version__)
print("Gradio:", gr.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("GPU not detected. Running on CPU.")


VERBA Environment
-----------------------------------
Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 4.57.6
SentencePiece: 0.2.2
Accelerate: 1.15.0
Gradio: 6.31.0
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB


In [44]:
# Load the pretrained NLLB-200 translation model

MODEL_NAME = "facebook/nllb-200-distilled-600M"
MODEL_PATH = "/content/drive/MyDrive/VERBA/models/nllb-200-distilled-600M"

MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Loading VERBA translation model from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE
)

model = model.to(DEVICE)
model.eval()

# Save the complete checkpoint to Google Drive
model.save_pretrained(MODEL_PATH, safe_serialization=True)
tokenizer.save_pretrained(MODEL_PATH)

print("\nVERBA model loaded and saved successfully!")
print("Parameters:", round(sum(p.numel() for p in model.parameters()) / 1_000_000, 2), "million")
print("Device:", DEVICE)
print("Precision:", MODEL_DTYPE)
print("Checkpoint:", MODEL_PATH)

Loading VERBA translation model from Hugging Face...


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



VERBA model loaded and saved successfully!
Parameters: 615.07 million
Device: cuda
Precision: torch.float16
Checkpoint: /content/drive/MyDrive/VERBA/models/nllb-200-distilled-600M


In [45]:
# Configure supported languages

LANGUAGES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Marathi": "mar_Deva",
    "Spanish": "spa_Latn",
    "Japanese": "jpn_Jpan"
}


# Core translation function

def translate(text, source_language, target_language):

    if not text or not text.strip():
        raise ValueError("Please enter text to translate.")

    if source_language not in LANGUAGES:
        raise ValueError("Unsupported source language.")

    if target_language not in LANGUAGES:
        raise ValueError("Unsupported target language.")

    if source_language == target_language:
        return text

    # Set the source language for NLLB
    tokenizer.src_lang = LANGUAGES[source_language]

    # Tokenize input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    # Get the target language token ID
    target_id = tokenizer.convert_tokens_to_ids(
        LANGUAGES[target_language]
    )

    # Generate translation
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_id,
            max_new_tokens=128,
            num_beams=4
        )

    # Convert generated tokens into readable text
    translated_text = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translated_text

In [46]:
# Demonstrate VERBA's multilingual translation capability

sample_text = "Welcome to Palghar. We are building a better future together."

target_languages = ["Hindi", "Marathi", "Spanish", "Japanese"]

print("VERBA Translation Demonstration")
print("-" * 50)
print("Source (English):", sample_text)

for language in target_languages:
    result = translate(sample_text, "English", language)

    print(f"\n{language}:")
    print(result)

VERBA Translation Demonstration
--------------------------------------------------
Source (English): Welcome to Palghar. We are building a better future together.

Hindi:
पालघर में आपका स्वागत है, हम एक साथ एक बेहतर भविष्य का निर्माण कर रहे हैं।

Marathi:
पालघरमध्ये आपले स्वागत आहे. आम्ही एकत्रितपणे एक चांगले भविष्य घडवत आहोत.

Spanish:
Bienvenidos a Palghar, estamos construyendo un futuro mejor juntos.

Japanese:
パルガーへようこそ 一緒により良い未来を築こう


In [47]:
# VERBA — Translation Performance Benchmark

evaluation_sentences = [
    "Welcome to Palghar.",
    "How are you today?",
    "Please bring your identity card.",
    "The hospital is located near the main road.",
    "We are building a better future together.",
    "Clean water is important for every community.",
    "The bus will arrive in ten minutes.",
    "Education creates new opportunities.",
    "Please contact us for more information.",
    "Thank you for your cooperation."
]

target_languages = ["Hindi", "Marathi", "Spanish", "Japanese"]

benchmark_results = []

# Warm-up run (not included in measurements)
translate("Welcome to Palghar.", "English", "Hindi")

if DEVICE == "cuda":
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

print("Running VERBA performance benchmark...")
print("Total translations:", len(evaluation_sentences) * len(target_languages))

for sentence in evaluation_sentences:
    for language in target_languages:

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        translated_text = translate(
            sentence,
            "English",
            language
        )

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        latency = time.perf_counter() - start_time

        benchmark_results.append({
            "source_text": sentence,
            "target_language": language,
            "translated_text": translated_text,
            "latency_seconds": round(latency, 4)
        })

# Calculate performance statistics
latencies = [
    result["latency_seconds"]
    for result in benchmark_results
]

# Save benchmark results
report_path = "/content/drive/MyDrive/VERBA/reports"

os.makedirs(report_path, exist_ok=True)

csv_path = os.path.join(
    report_path,
    "performance_benchmark.csv"
)

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=benchmark_results[0].keys()
    )
    writer.writeheader()
    writer.writerows(benchmark_results)

# Display summary
print("\nVERBA Performance Summary")
print("-" * 40)
print("Total translations:", len(latencies))
print("Average latency:", round(statistics.mean(latencies), 3), "seconds")
print("Median latency:", round(statistics.median(latencies), 3), "seconds")
print("Fastest translation:", round(min(latencies), 3), "seconds")
print("Slowest translation:", round(max(latencies), 3), "seconds")

for language in target_languages:
    language_times = [
        result["latency_seconds"]
        for result in benchmark_results
        if result["target_language"] == language
    ]

    print(
        f"{language}:",
        round(statistics.mean(language_times), 3),
        "seconds average"
    )

if DEVICE == "cuda":
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print("Peak GPU memory:", round(peak_memory, 3), "GB")

print("\nBenchmark report saved:", csv_path)

Running VERBA performance benchmark...
Total translations: 40

VERBA Performance Summary
----------------------------------------
Total translations: 40
Average latency: 0.335 seconds
Median latency: 0.322 seconds
Fastest translation: 0.164 seconds
Slowest translation: 0.922 seconds
Hindi: 0.329 seconds average
Marathi: 0.363 seconds average
Spanish: 0.35 seconds average
Japanese: 0.297 seconds average
Peak GPU memory: 2.349 GB

Benchmark report saved: /content/drive/MyDrive/VERBA/reports/performance_benchmark.csv


In [48]:
# VERBA — Multilingual Translation Interface

def run_translation(text, source_language, target_language):

    if not text or not text.strip():
        return "Please enter text to translate."

    try:
        return translate(text, source_language, target_language)

    except Exception as error:
        return f"Translation error: {str(error)}"


def swap_languages(source_language, target_language):
    return target_language, source_language


def clear_fields():
    return "", ""


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # VERBA
        ### Connecting Languages, Connecting People

        Translate text between English, Hindi, Marathi, Spanish and Japanese.
        """
    )

    with gr.Row():

        source_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="English",
            label="Translate From"
        )

        swap_button = gr.Button("⇄ Swap")

        target_language = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="Marathi",
            label="Translate To"
        )

    input_text = gr.Textbox(
        label="Enter Your Text",
        placeholder="Type something here...",
        lines=5
    )

    output_text = gr.Textbox(
        label="Translation",
        lines=5,
        interactive=False
    )

    with gr.Row():

        translate_button = gr.Button(
            "Translate",
            variant="primary"
        )

        clear_button = gr.Button("Clear")

    gr.Examples(
        examples=[
            ["Hello, how are you?", "English", "Hindi"],
            ["Welcome to Palghar.", "English", "Marathi"],
            ["Thank you for your help.", "English", "Spanish"],
            ["Good morning.", "English", "Japanese"]
        ],
        inputs=[input_text, source_language, target_language]
    )

    gr.Markdown(
        """
        **Disclaimer:** VERBA is an academic translation prototype.
        Please verify translations before using them in important situations.
        """
    )

    translate_button.click(
        fn=run_translation,
        inputs=[input_text, source_language, target_language],
        outputs=output_text
    )

    swap_button.click(
        fn=swap_languages,
        inputs=[source_language, target_language],
        outputs=[source_language, target_language]
    )

    clear_button.click(
        fn=clear_fields,
        inputs=[],
        outputs=[input_text, output_text]
    )


demo.launch(share=True)

/tmp/ipykernel_3794/2022339017.py:23: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4922cec1db46a26f2c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
